# Klasifikasi Gambar Bunga Menggunakan Random Forest

**Mata Kuliah:** Machine Learning  
**Dataset:** Gambar bunga (Bunga Melati Jakarta, Melati Jepang, Bintaro, Tapak Dara)  
**Metode UTS:** Random Forest  
**Total Data:** 1.440 gambar (4 kelas × 360 gambar)


## 1. Import Library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Preprocessing & Feature Extraction
from skimage.feature import hog
from skimage import color, exposure
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Visualisasi Confusion Matrix
from sklearn.metrics import ConfusionMatrixDisplay

import joblib

print("✅ Semua library berhasil diimport")

## 2. Konfigurasi Dataset

> **Sesuaikan `DATASET_DIR` dengan lokasi folder dataset kamu.**  
> Struktur folder yang diharapkan:
> ```
> dataset/
> ├── melati_jakarta/   (360 gambar)
> ├── melati_jepang/    (360 gambar)
> ├── bintaro/          (360 gambar)
> └── tapak_dara/       (360 gambar)
> ```

In [ ]:
# ============================================================
#  SESUAIKAN PATH INI DENGAN LOKASI DATASET KAMU
# ============================================================
DATASET_DIR = "dataset"   # Ganti jika lokasi berbeda

# Nama kelas (harus sama dengan nama folder di dalam DATASET_DIR)
CLASS_NAMES = ['melati_jakarta', 'melati_jepang', 'bintaro', 'tapak_dara']

# Parameter preprocessing gambar
IMG_SIZE    = (64, 64)   # Ukuran resize gambar
RANDOM_SEED = 42

# Split data
TEST_SIZE = 0.2   # 80% train, 20% test
VAL_SIZE  = 0.1   # 10% validasi dari data train

print(f"📁 Dataset directory : {DATASET_DIR}")
print(f"🌸 Kelas             : {CLASS_NAMES}")
print(f"📐 Ukuran gambar     : {IMG_SIZE}")
print(f"📊 Split             : Train 70% | Val 10% | Test 20%")

## 3. Eksplorasi Dataset (EDA)

In [ ]:
# ── Hitung jumlah gambar per kelas ──────────────────────────
class_counts = {}
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATASET_DIR, cls)
    count = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    class_counts[cls] = count

print("\n📊 Distribusi Dataset:")
print("-" * 35)
for cls, cnt in class_counts.items():
    print(f"  {cls:<20} : {cnt} gambar")
print("-" * 35)
print(f"  {'TOTAL':<20} : {sum(class_counts.values())} gambar")

In [ ]:
# ── Visualisasi distribusi dataset ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(class_counts.keys(), class_counts.values(),
                   color=['#4CAF50', '#2196F3', '#FF9800', '#E91E63'],
                   edgecolor='black', alpha=0.85)
axes[0].set_title('Distribusi Jumlah Data per Kelas', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Kelas Bunga')
axes[0].set_ylabel('Jumlah Gambar')
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels([c.replace('_', ' ').title() for c in class_counts.keys()],
                         rotation=15)
for bar, val in zip(bars, class_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values(),
            labels=[c.replace('_', ' ').title() for c in class_counts.keys()],
            autopct='%1.1f%%',
            colors=['#4CAF50', '#2196F3', '#FF9800', '#E91E63'],
            startangle=90)
axes[1].set_title('Proporsi Kelas Dataset', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('01_distribusi_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik distribusi dataset disimpan: 01_distribusi_dataset.png")

In [ ]:
# ── Tampilkan contoh gambar per kelas ───────────────────────
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Contoh Gambar Dataset per Kelas Bunga', fontsize=15, fontweight='bold')

for row_idx, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(DATASET_DIR, cls)
    images = [f for f in os.listdir(cls_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:5]
    for col_idx, img_file in enumerate(images):
        img = Image.open(os.path.join(cls_path, img_file)).convert('RGB')
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(
                cls.replace('_', ' ').title(), fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('02_contoh_gambar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Contoh gambar disimpan: 02_contoh_gambar.png")

## 4. Preprocessing Data

Tahapan preprocessing yang dilakukan:
1. **Load gambar** dari setiap folder kelas
2. **Resize** ke ukuran seragam 64×64 piksel
3. **Konversi ke grayscale** untuk ekstraksi fitur HOG
4. **Ekstraksi fitur HOG** (Histogram of Oriented Gradients)
5. **Normalisasi fitur** menggunakan StandardScaler
6. **Pembagian data** Train / Validation / Test

In [ ]:
def load_and_preprocess(dataset_dir, class_names, img_size):
    """
    Load gambar, resize, dan ekstrak fitur HOG.
    Mengembalikan array fitur X dan label y.
    """
    X, y = [], []
    for cls in class_names:
        cls_path = os.path.join(dataset_dir, cls)
        images = [f for f in os.listdir(cls_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"  Memproses [{cls}] ... {len(images)} gambar")
        for img_file in tqdm(images, desc=f"  {cls}", leave=False):
            try:
                img = Image.open(os.path.join(cls_path, img_file)).convert('RGB')
                img = img.resize(img_size)
                img_array = np.array(img)

                # Konversi ke grayscale untuk HOG
                img_gray = color.rgb2gray(img_array)

                # Ekstraksi fitur HOG
                hog_features = hog(
                    img_gray,
                    orientations=9,
                    pixels_per_cell=(8, 8),
                    cells_per_block=(2, 2),
                    visualize=False
                )

                # Fitur warna (mean & std tiap channel RGB)
                color_features = []
                for ch in range(3):
                    color_features.append(img_array[:, :, ch].mean() / 255.0)
                    color_features.append(img_array[:, :, ch].std()  / 255.0)

                # Gabung HOG + warna
                combined = np.concatenate([hog_features, color_features])
                X.append(combined)
                y.append(cls)
            except Exception as e:
                print(f"    ⚠️  Gagal load {img_file}: {e}")

    return np.array(X), np.array(y)


print("⏳ Memuat dan memproses dataset ...")
X, y = load_and_preprocess(DATASET_DIR, CLASS_NAMES, IMG_SIZE)
print(f"\n✅ Dataset berhasil dimuat")
print(f"   Jumlah sampel  : {X.shape[0]}")
print(f"   Jumlah fitur   : {X.shape[1]}")

In [ ]:
# ── Encoding label ───────────────────────────────────────────
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Pemetaan Label:")
for idx, cls in enumerate(le.classes_):
    print(f"  {idx} → {cls}")

In [ ]:
# ── Split: Train+Val (80%) vs Test (20%) ─────────────────────
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y_enc,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_enc
)

# Split: Train (87.5% dari trainval ≈ 70% total) vs Val (12.5% ≈ 10% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_SEED,
    stratify=y_trainval
)

print("📊 Pembagian Data:")
print(f"   Training   : {len(X_train)} sampel ({len(X_train)/len(X)*100:.1f}%)")
print(f"   Validasi   : {len(X_val)} sampel ({len(X_val)/len(X)*100:.1f}%)")
print(f"   Testing    : {len(X_test)} sampel ({len(X_test)/len(X)*100:.1f}%)")

In [ ]:
# ── Normalisasi (StandardScaler) ─────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("✅ Normalisasi fitur selesai (StandardScaler)")
print(f"   Mean fitur (train) : {X_train_scaled.mean():.4f}")
print(f"   Std fitur  (train) : {X_train_scaled.std():.4f}")

## 5. Visualisasi Contoh Fitur HOG

In [ ]:
# Visualisasi HOG untuk satu gambar per kelas
fig, axes = plt.subplots(4, 2, figsize=(10, 16))
fig.suptitle('Visualisasi Fitur HOG per Kelas', fontsize=14, fontweight='bold')

for row_idx, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(DATASET_DIR, cls)
    img_file = [f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))][0]
    img = Image.open(os.path.join(cls_path, img_file)).convert('RGB').resize(IMG_SIZE)
    img_gray = color.rgb2gray(np.array(img))

    hog_feats, hog_img = hog(
        img_gray, orientations=9,
        pixels_per_cell=(8, 8), cells_per_block=(2, 2),
        visualize=True
    )
    hog_img_rescaled = exposure.rescale_intensity(hog_img, in_range=(0, 10))

    axes[row_idx, 0].imshow(img)
    axes[row_idx, 0].set_title(f'Asli: {cls.replace("_", " ").title()}', fontsize=10)
    axes[row_idx, 0].axis('off')

    axes[row_idx, 1].imshow(hog_img_rescaled, cmap='gray')
    axes[row_idx, 1].set_title(f'HOG: {cls.replace("_", " ").title()}', fontsize=10)
    axes[row_idx, 1].axis('off')

plt.tight_layout()
plt.savefig('03_visualisasi_hog.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi HOG disimpan: 03_visualisasi_hog.png")

## 6. Pelatihan Model Random Forest

In [ ]:
# ── Inisialisasi dan training model ─────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,      # Jumlah pohon keputusan
    max_depth=None,        # Kedalaman pohon tidak dibatasi
    min_samples_split=2,   # Minimum sampel untuk split node
    min_samples_leaf=1,    # Minimum sampel di leaf node
    max_features='sqrt',   # Fitur yang dipertimbangkan tiap split
    class_weight='balanced',  # Menangani ketidakseimbangan kelas
    random_state=RANDOM_SEED,
    n_jobs=-1              # Gunakan semua core CPU
)

print("⏳ Melatih model Random Forest ...")
rf_model.fit(X_train_scaled, y_train)
print("✅ Training selesai!")

# Evaluasi pada data validasi
y_val_pred = rf_model.predict(X_val_scaled)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"\n📊 Akurasi Validasi : {val_accuracy*100:.2f}%")

## 7. Cross-Validation (K-Fold)

In [ ]:
# ── 5-Fold Stratified Cross-Validation ───────────────────────
# Menggunakan train+val untuk cross-validation
X_trainval_scaled = scaler.transform(X_trainval)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(
    rf_model, X_trainval_scaled, y_trainval,
    cv=skf, scoring='accuracy', n_jobs=-1
)

print("📊 Hasil Cross-Validation (5-Fold):")
for i, score in enumerate(cv_scores, 1):
    print(f"   Fold {i} : {score*100:.2f}%")
print(f"   {'─'*25}")
print(f"   Mean   : {cv_scores.mean()*100:.2f}%")
print(f"   Std    : {cv_scores.std()*100:.2f}%")

In [ ]:
# ── Visualisasi Cross-Validation ─────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
folds = [f'Fold {i}' for i in range(1, 6)]
bars = ax.bar(folds, cv_scores * 100,
              color='steelblue', edgecolor='black', alpha=0.85)
ax.axhline(cv_scores.mean() * 100, color='red', linestyle='--',
           linewidth=2, label=f'Rata-rata = {cv_scores.mean()*100:.2f}%')
ax.set_title('Hasil 5-Fold Cross-Validation - Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Fold')
ax.set_ylabel('Akurasi (%)')
ax.set_ylim(0, 110)
ax.legend()
for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val*100:.2f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('04_cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik cross-validation disimpan: 04_cross_validation.png")

## 8. Evaluasi Model pada Data Testing

In [ ]:
# ── Prediksi pada data test ───────────────────────────────────
y_test_pred = rf_model.predict(X_test_scaled)

# Hitung metrik evaluasi
acc  = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred, average='weighted')
rec  = recall_score(y_test, y_test_pred, average='weighted')
f1   = f1_score(y_test, y_test_pred, average='weighted')

print("=" * 45)
print("   HASIL EVALUASI MODEL - RANDOM FOREST")
print("=" * 45)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  Precision : {prec*100:.2f}%")
print(f"  Recall    : {rec*100:.2f}%")
print(f"  F1-Score  : {f1*100:.2f}%")
print("=" * 45)

In [ ]:
# ── Classification Report ────────────────────────────────────
class_labels = [c.replace('_', ' ').title() for c in le.classes_]
print("\n📋 Classification Report:")
print(classification_report(
    y_test, y_test_pred,
    target_names=class_labels
))

## 9. Visualisasi Hasil

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix - Random Forest', fontsize=13, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('05_confusion_matrix_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix disimpan: 05_confusion_matrix_rf.png")

In [ ]:
# ── Metrik per kelas (bar chart) ─────────────────────────────
precision_per_class = precision_score(y_test, y_test_pred, average=None)
recall_per_class    = recall_score(y_test, y_test_pred, average=None)
f1_per_class        = f1_score(y_test, y_test_pred, average=None)

x = np.arange(len(class_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width, precision_per_class * 100, width, label='Precision',
       color='#4CAF50', edgecolor='black', alpha=0.85)
ax.bar(x,          recall_per_class    * 100, width, label='Recall',
       color='#2196F3', edgecolor='black', alpha=0.85)
ax.bar(x + width,  f1_per_class        * 100, width, label='F1-Score',
       color='#FF9800', edgecolor='black', alpha=0.85)

ax.set_xlabel('Kelas Bunga')
ax.set_ylabel('Nilai (%)')
ax.set_title('Metrik Evaluasi per Kelas - Random Forest', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_labels, rotation=15)
ax.set_ylim(0, 115)
ax.legend()
ax.axhline(100, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('06_metrik_per_kelas_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik metrik per kelas disimpan: 06_metrik_per_kelas_rf.png")

In [ ]:
# ── Feature Importance ───────────────────────────────────────
importances = rf_model.feature_importances_
# Ambil 20 fitur teratas
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(range(top_n), importances[top_idx][::-1],
        color='steelblue', edgecolor='black', alpha=0.85)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f'Fitur {i}' for i in top_idx[::-1]])
ax.set_xlabel('Importance Score')
ax.set_title(f'Top {top_n} Feature Importance - Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('07_feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Feature importance disimpan: 07_feature_importance_rf.png")

In [ ]:
# ── Visualisasi contoh prediksi benar & salah ────────────────
# Ambil indeks sampel dari X_test (indeks asli dalam dataset)
# Kita perlu menyimpan nama file saat loading
# Tampilkan ringkasan prediksi saja jika gambar tidak tersedia

correct_idx   = np.where(y_test == y_test_pred)[0]
incorrect_idx = np.where(y_test != y_test_pred)[0]

print(f"✅ Prediksi BENAR : {len(correct_idx)} dari {len(y_test)} sampel")
print(f"❌ Prediksi SALAH : {len(incorrect_idx)} dari {len(y_test)} sampel")
print(f"   Akurasi Test   : {acc*100:.2f}%")

## 10. Ringkasan Hasil Model (Siap untuk Perbandingan dengan Deep Learning)

In [ ]:
# ── Simpan hasil ke DataFrame untuk perbandingan nanti ───────
rf_results = {
    'Model'    : 'Random Forest (ML)',
    'Accuracy' : round(acc  * 100, 2),
    'Precision': round(prec * 100, 2),
    'Recall'   : round(rec  * 100, 2),
    'F1-Score' : round(f1   * 100, 2),
    'CV Mean'  : round(cv_scores.mean() * 100, 2),
    'CV Std'   : round(cv_scores.std()  * 100, 2),
}

df_results = pd.DataFrame([rf_results])
print("\n📊 Ringkasan Hasil Random Forest:")
print(df_results.to_string(index=False))

# Simpan ke CSV
df_results.to_csv('hasil_rf.csv', index=False)
print("\n✅ Hasil disimpan ke: hasil_rf.csv")
print("   (CSV ini akan digunakan untuk tabel perbandingan dengan Deep Learning)")

In [ ]:
# ── Simpan model dan scaler ───────────────────────────────────
joblib.dump(rf_model, 'model_random_forest.pkl')
joblib.dump(scaler,   'scaler.pkl')
joblib.dump(le,       'label_encoder.pkl')

print("✅ Model disimpan:")
print("   → model_random_forest.pkl")
print("   → scaler.pkl")
print("   → label_encoder.pkl")

## 11. Demo Prediksi dengan Input Baru

In [ ]:
def predict_image(image_path, model, scaler, le, img_size=(64, 64)):
    """
    Prediksi kelas bunga dari satu gambar.
    
    Parameters
    ----------
    image_path : str  — path ke file gambar
    model      : trained RandomForestClassifier
    scaler     : fitted StandardScaler
    le         : fitted LabelEncoder
    img_size   : tuple (width, height)
    
    Returns
    -------
    dict berisi kelas prediksi dan probabilitas
    """
    # Load & preprocess
    img = Image.open(image_path).convert('RGB').resize(img_size)
    img_array = np.array(img)
    img_gray  = color.rgb2gray(img_array)

    hog_feats = hog(img_gray, orientations=9,
                    pixels_per_cell=(8, 8), cells_per_block=(2, 2),
                    visualize=False)

    color_feats = []
    for ch in range(3):
        color_feats.append(img_array[:, :, ch].mean() / 255.0)
        color_feats.append(img_array[:, :, ch].std()  / 255.0)

    features = np.concatenate([hog_feats, color_feats]).reshape(1, -1)
    features_scaled = scaler.transform(features)

    # Prediksi
    pred_idx   = model.predict(features_scaled)[0]
    pred_proba = model.predict_proba(features_scaled)[0]
    pred_label = le.inverse_transform([pred_idx])[0]

    # Tampilkan hasil
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].set_title(f'Gambar Input', fontsize=11)
    axes[0].axis('off')

    labels_display = [c.replace('_', ' ').title() for c in le.classes_]
    bars = axes[1].barh(labels_display, pred_proba * 100,
                        color=['#E91E63' if c == pred_label else 'steelblue'
                               for c in le.classes_],
                        edgecolor='black', alpha=0.85)
    axes[1].set_xlabel('Probabilitas (%)')
    axes[1].set_title(
        f'Prediksi: {pred_label.replace("_", " ").title()}\n'
        f'(Confidence: {max(pred_proba)*100:.1f}%)',
        fontsize=11, fontweight='bold'
    )
    axes[1].set_xlim(0, 110)
    for bar, val in zip(bars, pred_proba):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f'{val*100:.1f}%', va='center')

    plt.tight_layout()
    plt.savefig('08_demo_prediksi.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {'kelas': pred_label, 'probabilitas': dict(zip(le.classes_, pred_proba))}


# ── Contoh penggunaan ─────────────────────────────────────────
# Ganti path di bawah ini dengan gambar yang ingin diprediksi
# CONTOH_GAMBAR = "dataset/tapak_dara/img_001.jpg"
# hasil = predict_image(CONTOH_GAMBAR, rf_model, scaler, le)
# print(hasil)

print("✅ Fungsi predict_image siap digunakan.")
print("   Hapus komentar (#) pada baris CONTOH_GAMBAR di atas untuk mencoba.")

## 12. Kesimpulan

Bagian ini akan diisi setelah menjalankan semua sel di atas.

**Poin yang perlu disimpulkan:**
- Akurasi model Random Forest pada data testing
- Kelas bunga yang paling mudah/sulit diklasifikasikan
- Kelebihan dan kelemahan Random Forest untuk kasus ini
- Rencana pengembangan dengan Deep Learning (CNN/MobileNet)

In [ ]:
# ── Ringkasan Akhir ───────────────────────────────────────────
print("=" * 50)
print("         RINGKASAN HASIL - RANDOM FOREST")
print("=" * 50)
print(f" Dataset      : Gambar Bunga (4 kelas)")
print(f" Total Data   : {len(X)} sampel")
print(f" Fitur HOG    : {X.shape[1]} dimensi")
print(f" Split Data   : Train 70% | Val 10% | Test 20%")
print()
print(f" Accuracy     : {acc*100:.2f}%")
print(f" Precision    : {prec*100:.2f}%")
print(f" Recall       : {rec*100:.2f}%")
print(f" F1-Score     : {f1*100:.2f}%")
print(f" CV (5-Fold)  : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
print()
print(" Output Files :")
print("   01_distribusi_dataset.png")
print("   02_contoh_gambar.png")
print("   03_visualisasi_hog.png")
print("   04_cross_validation.png")
print("   05_confusion_matrix_rf.png")
print("   06_metrik_per_kelas_rf.png")
print("   07_feature_importance_rf.png")
print("   08_demo_prediksi.png")
print("   model_random_forest.pkl")
print("   hasil_rf.csv")
print("=" * 50)